# Core ML Sequential Experiments Runner

This notebook runs all attention variants with multiple positional encodings sequentially.
Results are logged to wandb and saved locally for comparison.

## Experiment Matrix
- **Attention variants:** vanilla_mha, sliding_window, gqa, relu_attention, sparse_attention, performer
- **Positional encodings:** sinusoidal, rope, alibi, relative
- **Total runs:** 6 variants × 4 positional = 24 experiments (configurable)

## Setup & Environment

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime

# Set working directory
os.chdir('/content/SAiDL-Summer-Assignment-2026') if os.path.exists('/content') else os.chdir('.')
sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")
print(f"Python version: {sys.version.split()[0]}")

## Define Experiment Grid

In [ ]:
# Attention variants to test
attention_variants = [
    "vanilla_mha",
    "sliding_window",
    "gqa",
    "relu_attention",
    "sparse_attention",
    "performer",
]

# Positional encodings to test
positional_variants = [
    "sinusoidal",
    "rope",
    "alibi",
    "relative",
]

# Total number of experiments
total_experiments = len(attention_variants) * len(positional_variants)
print(f"Total experiments to run: {total_experiments}")
print(f"Attention variants: {', '.join(attention_variants)}")
print(f"Positional encodings: {', '.join(positional_variants)}")

## Optional: Filter Experiments

Uncomment and modify to run a subset of experiments instead of all 24.

In [ ]:
# OPTIONAL: Run only a subset of experiments for quick testing
# Uncomment the line below to run only 2 experiments instead of 24

# attention_variants = ["vanilla_mha", "performer"]
# positional_variants = ["sinusoidal", "rope"]

total_experiments = len(attention_variants) * len(positional_variants)
print(f"Running {total_experiments} experiment(s)")

## Initialize Results Tracking

In [ ]:
import subprocess
from collections import defaultdict

# Results dictionary to store outcomes
results = {
    "timestamp": datetime.now().isoformat(),
    "total_experiments": total_experiments,
    "experiments": [],
    "summary": defaultdict(lambda: {"passed": 0, "failed": 0})
}

print(f"Experiment run started at {results['timestamp']}")

## Run Sequential Experiments

Each cell below runs one complete experiment with a specific attention + positional combination.
Monitor wandb logs in real-time or check the run names for each experiment.

In [ ]:
# Main experiment loop
experiment_count = 0

for attn in attention_variants:
    for pos in positional_variants:
        experiment_count += 1
        experiment_name = f"{attn}_{pos}"
        
        print(f"\n{'='*70}")
        print(f"Experiment {experiment_count}/{total_experiments}: {experiment_name}")
        print(f"{'='*70}")
        
        # Build the training command
        cmd = [
            "python",
            "core_ml/train/train.py",
            f"attention.name={attn}",
            f"positional.name={pos}",
            f"experiment_name={experiment_name}",
            "dataset.num_workers=0",  # Colab-friendly
        ]
        
        try:
            # Run the experiment
            result = subprocess.run(
                cmd,
                capture_output=False,
                timeout=3600,  # 1 hour timeout per experiment
            )
            
            if result.returncode == 0:
                status = "PASSED"
                results["summary"][attn]["passed"] += 1
            else:
                status = "FAILED"
                results["summary"][attn]["failed"] += 1
                
            results["experiments"].append({
                "experiment_name": experiment_name,
                "attention": attn,
                "positional": pos,
                "status": status,
                "index": experiment_count,
            })
            
            print(f"[{status}] Experiment {experiment_count} completed.")
            
        except subprocess.TimeoutExpired:
            print(f"[TIMEOUT] Experiment {experiment_count} exceeded 1 hour.")
            results["summary"][attn]["failed"] += 1
            results["experiments"].append({
                "experiment_name": experiment_name,
                "attention": attn,
                "positional": pos,
                "status": "TIMEOUT",
                "index": experiment_count,
            })
        except Exception as e:
            print(f"[ERROR] Experiment {experiment_count} failed with exception: {e}")
            results["summary"][attn]["failed"] += 1
            results["experiments"].append({
                "experiment_name": experiment_name,
                "attention": attn,
                "positional": pos,
                "status": f"ERROR: {str(e)[:50]}",
                "index": experiment_count,
            })

print(f"\n{'='*70}")
print("All experiments completed!")
print(f"{'='*70}")

## Results Summary

In [ ]:
import pandas as pd

# Convert results to DataFrame for easy viewing
df_results = pd.DataFrame(results["experiments"])

print("\nEXPERIMENT RESULTS SUMMARY")
print("="*70)
print(df_results.to_string(index=False))

# Summary by attention variant
print("\n" + "="*70)
print("SUMMARY BY ATTENTION VARIANT")
print("="*70)

summary_data = []
for attn in attention_variants:
    passed = results["summary"][attn]["passed"]
    failed = results["summary"][attn]["failed"]
    total = passed + failed
    success_rate = (passed / total * 100) if total > 0 else 0
    summary_data.append({
        "Attention": attn,
        "Passed": passed,
        "Failed": failed,
        "Total": total,
        "Success Rate": f"{success_rate:.1f}%"
    })

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

# Overall statistics
total_passed = sum(r["summary"][attn]["passed"] for attn in attention_variants)
total_failed = sum(r["summary"][attn]["failed"] for attn in attention_variants)
print(f"\nTotal Passed: {total_passed}/{total_experiments}")
print(f"Total Failed: {total_failed}/{total_experiments}")
print(f"Overall Success Rate: {total_passed/total_experiments*100:.1f}%")

## Save Results to File

In [ ]:
import json

# Save results as JSON
results_file = "core_ml_experiments_results.json"
with open(results_file, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {results_file}")

# Also save as CSV for easy spreadsheet viewing
df_results.to_csv("core_ml_experiments_results.csv", index=False)
print(f"CSV saved to core_ml_experiments_results.csv")

## WandB Links

All experiments are logged to WandB. Visit:
https://wandb.ai/your-username/SAiDL-Core-ML

Look for run names matching the pattern: `{attention_variant}_{positional_encoding}`

## Notes

- Each experiment runs with the optimized Colab T4 settings (seq_len=1024, d_model=256, batch_size=8, num_workers=0)
- Set `num_workers=2` in the command if running on a multi-core CPU/GPU machine
- Performer with causal masking will raise NotImplementedError (requires custom kernels)
- To run a subset of experiments, modify the `attention_variants` and `positional_variants` lists above